# AO3 Tag Visualizer

Notebook version of `ao3_tag_visualizer.py`. Reads an existing `ao3_tag_metadata.csv`
(produced by `ao3_tag_scraper.py`/`.ipynb`) and visualizes connections between the seed
tag (the popular tag from `archiveofourown.org/tags` used as the search term) and each
work's `rating`, `warnings`, `category`, `fandom`, and `additional_tags`:

- an **interactive bipartite network graph** (seed tags <-> attribute values, edges
  weighted by co-occurrence count)
- a **co-occurrence heatmap per field** (rows = seed tags, columns = attribute values)

`fandom` and `additional_tags` are high-cardinality, so each is filtered to its top-N
most frequent values (computed from the full dataset) before either visualization is
built.

**No network access is required** -- this only reads a local CSV.

The first code cell installs this notebook's dependencies (pandas, networkx, pyvis, matplotlib, seaborn) if they're missing -- safe to re-run.

Edit the Configuration cell, then run all cells in order.

**Kernel restart required after updating:** if you've previously run an older version
of this notebook in the same session, do **Kernel -> Restart Kernel and Run All Cells**.
Stale function definitions from a previous run persist in memory until the kernel is
restarted.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present
# (pandas, networkx, pyvis, matplotlib, seaborn, scipy). Safe to re-run --
# already satisfied requirements are a fast no-op. Restart the kernel
# afterward if this is the first time you've run this cell in the current
# session.
%pip install -q pandas networkx pyvis matplotlib seaborn scipy

In [ ]:
import json
import os
import re

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from pyvis.network import Network
from pyvis.node import Node
from pyvis.edge import Edge

## Configuration

Edit these values, then run all cells in order.

In [ ]:
INPUT = "ao3_tag_metadata.csv"
TOP_FANDOMS = 20              # keep only the top N most frequent fandoms overall
TOP_ADDITIONAL_TAGS = 40      # keep only the top N most frequent additional tags overall
MIN_COUNT = 2                 # drop edges/cells below this co-occurrence count
MIN_PROPORTION = None         # alternative to MIN_COUNT: drop a (seed tag, value)
                              # edge/cell below this fraction (0.0-1.0) of that
                              # seed tag's total works. Set exactly one of
                              # MIN_COUNT / MIN_PROPORTION to a value, leave the
                              # other as None
TOP_SEED_TAGS = None          # limit to the N seed tags with the most works (None = all)
NETWORK_OUT = "ao3_tag_network.html"
HEATMAP_OUT_DIR = "heatmaps"

DELIMITER = ", "
FIELDS_TO_VISUALIZE = ["rating", "warnings", "category", "fandom", "additional_tags"]
FIELDS_TOP_N_ELIGIBLE = {"fandom", "additional_tags"}
FIELD_COLORS = {
    "seed_tag": "#4C72B0", "rating": "#DD8452", "warnings": "#55A868",
    "category": "#C44E52", "fandom": "#8172B2", "additional_tags": "#937860",
    "character": "#CCB974", "relationship": "#64B5CD",
}
assert MIN_COUNT is None or MIN_PROPORTION is None, \
    "Set only one of MIN_COUNT / MIN_PROPORTION (leave the other as None)"

# Tag-pair co-occurrence (lift/PMI) -- see the "Tag-pair co-occurrence" section below
COMPUTE_TAG_PAIRS = False      # set True to also compute/render tag-pair stats
TOP_TAGS = 40                  # top N tags overall, by document frequency
MIN_PAIR_COUNT = 2             # drop pairs with fewer co-occurrences than this
MIN_PMI = 1.0                  # "most likely" threshold: keep pairs with pmi >= this
MAX_PMI = -1.0                 # "least likely" threshold: keep pairs with pmi <= this
TAG_PAIR_NETWORK_OUT = "ao3_tag_pair_network.html"
TAG_PAIR_HEATMAP_OUT = "heatmaps/heatmap_tag_pairs.png"
TAG_PAIR_FIELDS = ["fandom", "relationship", "character", "additional_tags"]
MOST_LIKELY_EDGE_COLOR = "#2A9D8F"    # pair co-occurs more often than chance
LEAST_LIKELY_EDGE_COLOR = "#E63946"   # pair co-occurs less often than chance
if MIN_PMI <= MAX_PMI:
    print(f"  warning: MIN_PMI ({MIN_PMI}) <= MAX_PMI ({MAX_PMI}) -- "
          "these bands overlap and will include nearly all pairs")


## Load and explode metadata

The CSV is loaded once. Multi-value fields (e.g. `additional_tags`, `category`) are joined with `", "` in the scraper's output, so they're split back out into one row per distinct value per work -- this is what makes a work with 3 additional tags contribute 3 separate co-occurrence increments, not 1.

In [ ]:
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    if df.empty:
        raise ValueError(f"{input_csv} has no data rows.")
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    values = [v.strip() for v in cell.split(delimiter) if v.strip()]
    return list(dict.fromkeys(values))


def explode_field(df, field):
    exploded = df[["tag", "work_id", field]].copy()
    exploded[field] = exploded[field].map(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


df = load_metadata(INPUT)
df.head()

## Co-occurrence counting and top-N filtering

`fandom` and `additional_tags` are filtered to their top-N most frequent values, computed from the *full* dataset before any per-seed-tag filtering -- so a kept value's count always reflects its true frequency, not a reduced one.

In [ ]:
def total_value_counts(exploded, field):
    return exploded[field].value_counts()


def top_n_values(exploded, field, n):
    if n is None:
        return None
    counts = total_value_counts(exploded, field).sort_values(ascending=False)
    counts = counts.reset_index()
    counts.columns = [field, "count"]
    counts = counts.sort_values(["count", field], ascending=[False, True])
    return set(counts[field].head(n))


def cooccurrence_counts(exploded, field, keep_values):
    if keep_values is not None:
        exploded = exploded[exploded[field].isin(keep_values)]
    if exploded.empty:
        return pd.DataFrame(columns=["tag", field, "count"])
    return exploded.groupby(["tag", field]).size().reset_index(name="count")


def apply_min_count(counts, min_count):
    return counts[counts["count"] >= min_count]


def apply_min_proportion(counts, work_counts, min_proportion):
    """Alternative to apply_min_count: keeps a (tag, value) pair only if it
    appears in at least min_proportion (0.0-1.0) of that seed tag's own
    total works, rather than by a raw absolute count."""
    denom = counts["tag"].map(work_counts)
    return counts[(counts["count"] / denom) >= min_proportion]


def rank_seed_tags(df, top_seed_tags):
    ranked = df["tag"].value_counts().index.tolist()
    return ranked if top_seed_tags is None else ranked[:top_seed_tags]


def build_field_data(df, top_fandoms, top_additional_tags, work_counts=None,
                      min_count=None, min_proportion=None):
    top_n_by_field = {"fandom": top_fandoms, "additional_tags": top_additional_tags}
    field_tables = {}
    for field in FIELDS_TO_VISUALIZE:
        exploded = explode_field(df, field)
        n = top_n_by_field.get(field) if field in FIELDS_TOP_N_ELIGIBLE else None
        keep_values = top_n_values(exploded, field, n)
        counts = cooccurrence_counts(exploded, field, keep_values)
        if min_proportion is not None:
            counts = apply_min_proportion(counts, work_counts, min_proportion)
        else:
            counts = apply_min_count(counts, min_count)
        field_tables[field] = counts
    return field_tables


# Needed unconditionally: heatmap cells are always colored/labeled by
# percentage of each seed tag's works (not raw count), and MIN_PROPORTION
# filtering needs the same per-tag totals when active.
work_counts = df["tag"].value_counts()
seed_tags = rank_seed_tags(df, TOP_SEED_TAGS)
field_tables = build_field_data(df, TOP_FANDOMS, TOP_ADDITIONAL_TAGS,
                                 work_counts=work_counts,
                                 min_count=MIN_COUNT, min_proportion=MIN_PROPORTION)
{field: len(counts) for field, counts in field_tables.items()}  # preview: surviving edges per field

## Interactive network graph

Nodes: seed tags (blue) + attribute values (one color per field). Edges: co-occurrence count (hover for the exact count). Pan/zoom/drag are built in; no internet connection is needed to view the resulting HTML file.

In [ ]:
def build_bipartite_graph(field_tables, seed_tags):
    graph = nx.Graph()
    seed_tag_set = set(seed_tags)
    for tag in seed_tags:
        graph.add_node(f"tag::{tag}", label=tag, group="seed_tag",
                        color=FIELD_COLORS["seed_tag"], title=tag)

    for field, counts in field_tables.items():
        for _, row in counts.iterrows():
            tag, value, count = row["tag"], row[field], row["count"]
            if tag not in seed_tag_set:
                continue
            value_id = f"{field}::{value}"
            if value_id not in graph:
                graph.add_node(value_id, label=value, group=field,
                                color=FIELD_COLORS[field], title=f"{field}: {value}")
            graph.add_edge(f"tag::{tag}", value_id, weight=int(count),
                            title=f"{tag} × {value}: {count} works")
    return graph


_BOOTSTRAP_TAG_RE = re.compile(
    r'<(?:link|script)[^>]*\bhttps://cdn\.jsdelivr\.net/npm/bootstrap[^>]*>(?:</script>)?\s*'
)


def _strip_bootstrap_cdn(html_path):
    # pyvis's bundled template.html unconditionally references Bootstrap from
    # a CDN for the "card"/"card-body" wrapper div, regardless of
    # cdn_resources (which only inlines vis-network's own JS/CSS). That div
    # is purely cosmetic -- the graph itself is vis-network, already inlined,
    # and has no Bootstrap dependency -- so strip the two tags to make the
    # file genuinely self-contained.
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    html = _BOOTSTRAP_TAG_RE.sub("", html)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


def _seed_tags_from_graph(graph):
    """Recovers the seed-tag label list, in rank order, from a graph built by
    build_bipartite_graph. Seed-tag nodes are added first there, and
    networkx/dict node storage preserves insertion order, so this avoids
    threading seed_tags through render_network() as a separate parameter."""
    return [data["label"] for _, data in graph.nodes(data=True)
            if data.get("group") == "seed_tag"]


_FILTER_PANEL_CSS = """
<style>
#ao3-filter-panel { font-family: sans-serif; font-size: 14px; padding: 10px 14px;
  border-bottom: 1px solid #ccc; background: #fafafa; }
#ao3-filter-panel .ao3-cat-label { margin-right: 14px; white-space: nowrap; }
#ao3-filter-panel .ao3-swatch { display: inline-block; width: 10px; height: 10px;
  border-radius: 50%; margin: 0 4px 0 4px; vertical-align: middle; }
#ao3-tag-picker { margin-top: 8px; position: relative; }
#ao3-tag-picker .ao3-hint { color: #777; font-weight: normal; font-size: 12px; }
#ao3-tag-chips { display: flex; flex-wrap: wrap; gap: 4px; margin: 6px 0; }
.ao3-tag-chip { background: #4C72B0; color: #fff; border-radius: 12px;
  padding: 2px 6px 2px 10px; font-size: 12px; display: inline-flex; align-items: center; }
.ao3-chip-remove { background: none; border: none; color: #fff; cursor: pointer;
  font-size: 14px; margin-left: 4px; line-height: 1; }
#ao3-tag-search { width: 260px; padding: 4px 6px; }
#ao3-tag-dropdown { position: absolute; z-index: 10; background: #fff; border: 1px solid #ccc;
  max-height: 220px; overflow-y: auto; width: 260px; }
.ao3-tag-option { padding: 4px 8px; cursor: pointer; }
.ao3-tag-option:hover { background: #eef; }
</style>
"""

_FILTER_PANEL_HTML = """
<div id="ao3-filter-panel">
  <div id="ao3-cat-checkboxes">__CHECKBOX_ITEMS__</div>
  <div id="ao3-tag-picker">
    <strong>Seed tags</strong> <span class="ao3-hint">(none selected = show all)</span>
    <div id="ao3-tag-chips"></div>
    <input type="text" id="ao3-tag-search" placeholder="Search seed tags…" autocomplete="off">
    <div id="ao3-tag-dropdown" hidden></div>
  </div>
</div>
""" + _FILTER_PANEL_CSS

_FILTER_SCRIPT_TEMPLATE = """
<script>
(function () {
  const ALL_SEED_TAGS = __SEED_TAGS_JSON__;
  let selectedTags = [];

  function buildNodeGroups() {
    // Built from our own nodes.get() call rather than reusing any of
    // pyvis's own internal globals -- nodes/edges/network are the stable
    // vis-network API surface, not an implementation detail.
    const map = {};
    for (const n of nodes.get()) map[n.id] = n.group;
    return map;
  }

  function buildSeedAdjacency() {
    // seed-tag-node-id -> Set of connected attribute-value-node-ids, built
    // once so applyFilters() never has to rescan every edge.
    const adj = {};
    for (const e of edges.get()) {
      const aSeed = String(e.from).startsWith("tag::");
      const bSeed = String(e.to).startsWith("tag::");
      if (aSeed && !bSeed) { (adj[e.from] = adj[e.from] || new Set()).add(e.to); }
      else if (bSeed && !aSeed) { (adj[e.to] = adj[e.to] || new Set()).add(e.from); }
    }
    return adj;
  }

  const NODE_GROUPS = buildNodeGroups();
  const SEED_ADJACENCY = buildSeedAdjacency();

  function applyFilters() {
    const checkedGroups = new Set(
      Array.from(document.querySelectorAll('.ao3-cat-checkbox:checked'))
           .map(function (cb) { return cb.dataset.group; })
    );
    const seedLabels = selectedTags.length ? selectedTags : ALL_SEED_TAGS;
    const visibleSeedIds = new Set(seedLabels.map(function (t) { return "tag::" + t; }));

    const reachableAttrIds = new Set();
    visibleSeedIds.forEach(function (seedId) {
      const neighbors = SEED_ADJACENCY[seedId];
      if (neighbors) neighbors.forEach(function (attrId) { reachableAttrIds.add(attrId); });
    });

    const updates = [];
    for (const nodeId in NODE_GROUPS) {
      const group = NODE_GROUPS[nodeId];
      let hidden;
      if (group === "seed_tag") {
        hidden = !visibleSeedIds.has(nodeId);
      } else {
        hidden = !(checkedGroups.has(group) && reachableAttrIds.has(nodeId));
      }
      updates.push({ id: nodeId, hidden: hidden });
    }
    nodes.update(updates); // vis-network auto-hides edges with a hidden endpoint
  }

  function matchingTags(query) {
    // Empty query -> the full remaining list (not []), so clicking/focusing
    // the search box opens the dropdown with every not-yet-selected tag,
    // not just typed-in matches. The scrollable dropdown (max-height in CSS)
    // handles a full ~200-tag list fine, so no arbitrary result cap either.
    const q = query.trim().toLowerCase();
    const available = ALL_SEED_TAGS.filter(function (t) { return selectedTags.indexOf(t) === -1; });
    if (!q) return available;
    return available.filter(function (t) { return t.toLowerCase().indexOf(q) !== -1; });
  }

  function renderDropdown(query) {
    const dropdown = document.getElementById('ao3-tag-dropdown');
    dropdown.innerHTML = "";
    const matches = matchingTags(query);
    matches.forEach(function (tag) {
      const opt = document.createElement('div');
      opt.className = 'ao3-tag-option';
      opt.textContent = tag;
      opt.dataset.tag = tag;
      dropdown.appendChild(opt);
    });
    dropdown.hidden = matches.length === 0;
  }

  function renderChips() {
    const chips = document.getElementById('ao3-tag-chips');
    chips.innerHTML = "";
    selectedTags.forEach(function (tag) {
      const chip = document.createElement('span');
      chip.className = 'ao3-tag-chip';
      const label = document.createElement('span');
      label.textContent = tag;
      const remove = document.createElement('button');
      remove.className = 'ao3-chip-remove';
      remove.textContent = '\\u00d7';
      remove.dataset.tag = tag;
      chip.appendChild(label);
      chip.appendChild(remove);
      chips.appendChild(chip);
    });
  }

  function addTag(tag) {
    if (selectedTags.indexOf(tag) === -1) selectedTags.push(tag);
    renderChips();
    applyFilters();
  }

  function removeTag(tag) {
    selectedTags = selectedTags.filter(function (t) { return t !== tag; });
    renderChips();
    applyFilters();
  }

  const searchInput = document.getElementById('ao3-tag-search');
  searchInput.addEventListener('input', function () { renderDropdown(searchInput.value); });
  // Open the full tag list on focus/click too, not just once the user has
  // typed something -- gives a visible "click to open" affordance like a
  // normal dropdown, instead of only reacting to keystrokes.
  searchInput.addEventListener('focus', function () { renderDropdown(searchInput.value); });
  searchInput.addEventListener('click', function () { renderDropdown(searchInput.value); });

  document.getElementById('ao3-tag-dropdown').addEventListener('click', function (e) {
    const opt = e.target.closest('.ao3-tag-option');
    if (!opt) return;
    addTag(opt.dataset.tag);
    searchInput.value = "";
    document.getElementById('ao3-tag-dropdown').hidden = true;
  });

  document.getElementById('ao3-tag-chips').addEventListener('click', function (e) {
    const btn = e.target.closest('.ao3-chip-remove');
    if (btn) removeTag(btn.dataset.tag);
  });

  document.addEventListener('click', function (e) {
    const dropdown = document.getElementById('ao3-tag-dropdown');
    if (!dropdown.contains(e.target) && e.target !== searchInput) dropdown.hidden = true;
  });

  document.querySelectorAll('.ao3-cat-checkbox').forEach(function (cb) {
    cb.addEventListener('change', applyFilters);
  });

  applyFilters();
})();
</script>
"""


def _filter_controls_html(graph):
    """Builds the injected filter-UI markup and script as a (panel_html,
    script_html) pair. Pure string building, no file I/O."""
    seed_tags = _seed_tags_from_graph(graph)
    # A literal "</script" substring in the JSON would prematurely close the
    # <script> tag at the HTML-tokenizer level regardless of correct
    # JS/JSON escaping -- guard against it independent of AO3 data.
    seed_tags_json = json.dumps(seed_tags).replace("</script", "<\\/script")

    checkbox_items = "\n".join(
        '<label class="ao3-cat-label">'
        f'<input type="checkbox" class="ao3-cat-checkbox" data-group="{field}" checked>'
        f'<span class="ao3-swatch" style="background-color:{FIELD_COLORS[field]};"></span>'
        f'{field}</label>'
        for field in FIELDS_TO_VISUALIZE
    )
    # FIELDS_TO_VISUALIZE / FIELD_COLORS are hardcoded module constants
    # (lowercase ascii identifiers, hex colors), never user data, so no
    # HTML-escaping is needed here. Seed-tag labels (arbitrary AO3 text) are
    # never interpolated into an HTML string anywhere in this function --
    # they only ever travel through the JSON-encoded ALL_SEED_TAGS array,
    # and the client renders them via textContent/dataset, never innerHTML.
    panel_html = _FILTER_PANEL_HTML.replace("__CHECKBOX_ITEMS__", checkbox_items)
    script_html = _FILTER_SCRIPT_TEMPLATE.replace("__SEED_TAGS_JSON__", seed_tags_json)
    return panel_html, script_html


def _inject_filter_controls(html_path, graph):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    # pyvis's template always emits exactly one bare <body>/</body> pair;
    # assert instead of silently no-oping if a future pyvis version changes
    # this, so template drift fails loudly rather than shipping a graph with
    # no filter UI.
    assert html.count("<body>") == 1, "expected exactly one <body> tag"
    assert html.count("</body>") == 1, "expected exactly one </body> tag"
    panel_html, script_html = _filter_controls_html(graph)
    html = html.replace("<body>", "<body>" + panel_html, 1)
    html = html.replace("</body>", script_html + "</body>", 1)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


def _all_tags_from_graph(graph):
    """Recovers [{"id", "label", "field"}] for every node, in insertion
    order, for the tag-pair picker. Unlike the bipartite picker (which
    stores plain labels, since seed tags can't collide with each other),
    this stores the full namespaced node id as the selection key -- label
    alone is exactly what CAN collide across fields (e.g. a fandom and a
    character sharing a literal name)."""
    return [{"id": node_id, "label": data["label"], "field": data["group"]}
            for node_id, data in graph.nodes(data=True)]


_TAG_PAIR_FILTER_PANEL_HTML = """
<div id="ao3-filter-panel">
  <div id="ao3-cat-checkboxes">__CHECKBOX_ITEMS__</div>
  <div id="ao3-tag-picker">
    <strong>Tags</strong> <span class="ao3-hint">(none selected = show all;
    selecting narrows to those tags + their direct connections)</span>
    <div id="ao3-tag-chips"></div>
    <input type="text" id="ao3-tag-search" placeholder="Search tags…" autocomplete="off">
    <div id="ao3-tag-dropdown" hidden></div>
  </div>
</div>
""" + _FILTER_PANEL_CSS

_TAG_PAIR_FILTER_SCRIPT_TEMPLATE = """
<script>
(function () {
  const ALL_TAGS = __ALL_TAGS_JSON__;
  let selectedTagIds = [];

  function buildNodeGroups() {
    const map = {};
    for (const n of nodes.get()) map[n.id] = n.group;
    return map;
  }

  function buildFullAdjacency() {
    // Every node here is a "tag" -- unlike the bipartite graph, there's no
    // privileged always-visible node class, so adjacency is symmetric and
    // built for every node, not just one side of a bipartite split.
    const adj = {};
    for (const e of edges.get()) {
      (adj[e.from] = adj[e.from] || new Set()).add(e.to);
      (adj[e.to] = adj[e.to] || new Set()).add(e.from);
    }
    return adj;
  }

  const NODE_GROUPS = buildNodeGroups();
  const FULL_ADJACENCY = buildFullAdjacency();

  function applyFilters() {
    const checkedGroups = new Set(
      Array.from(document.querySelectorAll('.ao3-cat-checkbox:checked'))
           .map(function (cb) { return cb.dataset.group; })
    );
    let reachable = null;
    if (selectedTagIds.length) {
      reachable = new Set();
      selectedTagIds.forEach(function (id) {
        reachable.add(id);
        const neighbors = FULL_ADJACENCY[id];
        if (neighbors) neighbors.forEach(function (n) { reachable.add(n); });
      });
    }
    const updates = [];
    for (const nodeId in NODE_GROUPS) {
      let hidden = !checkedGroups.has(NODE_GROUPS[nodeId]);
      if (!hidden && reachable) hidden = !reachable.has(nodeId);
      updates.push({ id: nodeId, hidden: hidden });
    }
    nodes.update(updates); // vis-network auto-hides edges with a hidden endpoint
  }

  function tagById(id) {
    return ALL_TAGS.find(function (t) { return t.id === id; });
  }

  function matchingTags(query) {
    const q = query.trim().toLowerCase();
    const available = ALL_TAGS.filter(function (t) { return selectedTagIds.indexOf(t.id) === -1; });
    if (!q) return available;
    return available.filter(function (t) {
      return t.label.toLowerCase().indexOf(q) !== -1 || t.field.toLowerCase().indexOf(q) !== -1;
    });
  }

  function renderDropdown(query) {
    const dropdown = document.getElementById('ao3-tag-dropdown');
    dropdown.innerHTML = "";
    const matches = matchingTags(query);
    matches.forEach(function (t) {
      const opt = document.createElement('div');
      opt.className = 'ao3-tag-option';
      opt.textContent = t.label + " (" + t.field + ")";
      opt.dataset.id = t.id;
      dropdown.appendChild(opt);
    });
    dropdown.hidden = matches.length === 0;
  }

  function renderChips() {
    const chips = document.getElementById('ao3-tag-chips');
    chips.innerHTML = "";
    selectedTagIds.forEach(function (id) {
      const t = tagById(id);
      const chip = document.createElement('span');
      chip.className = 'ao3-tag-chip';
      const label = document.createElement('span');
      label.textContent = t ? (t.label + " (" + t.field + ")") : id;
      const remove = document.createElement('button');
      remove.className = 'ao3-chip-remove';
      remove.textContent = '\u00d7';
      remove.dataset.id = id;
      chip.appendChild(label);
      chip.appendChild(remove);
      chips.appendChild(chip);
    });
  }

  function addTag(id) {
    if (selectedTagIds.indexOf(id) === -1) selectedTagIds.push(id);
    renderChips();
    applyFilters();
  }

  function removeTag(id) {
    selectedTagIds = selectedTagIds.filter(function (x) { return x !== id; });
    renderChips();
    applyFilters();
  }

  const searchInput = document.getElementById('ao3-tag-search');
  searchInput.addEventListener('input', function () { renderDropdown(searchInput.value); });
  searchInput.addEventListener('focus', function () { renderDropdown(searchInput.value); });
  searchInput.addEventListener('click', function () { renderDropdown(searchInput.value); });

  document.getElementById('ao3-tag-dropdown').addEventListener('click', function (e) {
    const opt = e.target.closest('.ao3-tag-option');
    if (!opt) return;
    addTag(opt.dataset.id);
    searchInput.value = "";
    document.getElementById('ao3-tag-dropdown').hidden = true;
  });

  document.getElementById('ao3-tag-chips').addEventListener('click', function (e) {
    const btn = e.target.closest('.ao3-chip-remove');
    if (btn) removeTag(btn.dataset.id);
  });

  document.addEventListener('click', function (e) {
    const dropdown = document.getElementById('ao3-tag-dropdown');
    if (!dropdown.contains(e.target) && e.target !== searchInput) dropdown.hidden = true;
  });

  document.querySelectorAll('.ao3-cat-checkbox').forEach(function (cb) {
    cb.addEventListener('change', applyFilters);
  });

  applyFilters();
})();
</script>
"""


def _tag_pair_filter_controls_html(graph):
    """Same shape as _filter_controls_html, for the one-mode tag-pair graph:
    checkboxes iterate TAG_PAIR_FIELDS (every node's group is checkbox-
    controlled -- there's no privileged always-visible class like
    seed_tag), and the tag picker's underlying data is
    [{"id","label","field"}, ...] rather than a flat label list, since
    label alone can collide across fields."""
    all_tags = _all_tags_from_graph(graph)
    all_tags_json = json.dumps(all_tags).replace("</script", "<\/script")

    checkbox_items = "\n".join(
        '<label class="ao3-cat-label">'
        f'<input type="checkbox" class="ao3-cat-checkbox" data-group="{field}" checked>'
        f'<span class="ao3-swatch" style="background-color:{FIELD_COLORS[field]};"></span>'
        f'{field}</label>'
        for field in TAG_PAIR_FIELDS
    )
    panel_html = _TAG_PAIR_FILTER_PANEL_HTML.replace("__CHECKBOX_ITEMS__", checkbox_items)
    script_html = _TAG_PAIR_FILTER_SCRIPT_TEMPLATE.replace("__ALL_TAGS_JSON__", all_tags_json)
    return panel_html, script_html


def _inject_tag_pair_filter_controls(html_path, graph):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    assert html.count("<body>") == 1, "expected exactly one <body> tag"
    assert html.count("</body>") == 1, "expected exactly one </body> tag"
    panel_html, script_html = _tag_pair_filter_controls_html(graph)
    html = html.replace("<body>", "<body>" + panel_html, 1)
    html = html.replace("</body>", script_html + "</body>", 1)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


# layout/physics (including improvedLayout and avoidOverlap) are configured
# at network-construction time via Network.set_options() -- see
# _NETWORK_OPTIONS_JSON below -- not here. A later network.setOptions() call
# (this script only runs once the page loads, after the network already
# exists) is too late to prevent vis-network's improvedLayout initial
# positioning attempt, which can fail outright on graphs like this one
# ("This network could not be positioned by this version of the improved
# layout algorithm") and silently stall stabilization for a long time.
_STABILIZE_THEN_STOP_SCRIPT = """
<script>
(function () {
  // Physics is configured to run (with overlap avoidance) until the layout
  // settles; once it does, turn physics off entirely so the graph freezes
  // instead of drifting/jittering indefinitely.
  network.once("stabilizationIterationsDone", function () {
    network.setOptions({ physics: false });
  });
})();
</script>
"""


def _inject_stabilize_then_stop(html_path):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    assert "</body>" in html, "expected a </body> tag"
    html = html.replace("</body>", _STABILIZE_THEN_STOP_SCRIPT + "</body>", 1)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


# Passed to Network.set_options() before write_html() -- must be strict
# JSON (pyvis's Options.set() does a bare json.loads on it, no JS-style
# comments/trailing commas/unquoted keys). avoidOverlap spreads nodes out
# enough to reduce label collisions once physics stabilizes (see
# _STABILIZE_THEN_STOP_SCRIPT); improvedLayout is disabled because vis-network's
# own clustering-based initial positioner can fail outright on graphs like
# this one and silently stall stabilization; iterations is capped low so
# stabilization can't run away indefinitely on a large/dense graph.
_NETWORK_OPTIONS_JSON = json.dumps({
    "layout": {"improvedLayout": False},
    "physics": {
        "solver": "barnesHut",
        "barnesHut": {"avoidOverlap": 0.5},
        "stabilization": {"enabled": True, "iterations": 200, "fit": True},
    },
})

# Used when the caller has already baked fixed x/y positions into every
# node -- physics off means vis-network places nodes at exactly those
# coordinates and never runs a client-side force simulation over them.
_STATIC_NETWORK_OPTIONS_JSON = json.dumps({
    "layout": {"improvedLayout": False},
    "physics": {"enabled": False},
})


def _fast_populate_network(net, graph, default_node_size=10, default_edge_weight=1):
    # Equivalent to pyvis's Network.from_nx(graph) -- confirmed to produce
    # byte-identical net.nodes/net.edges/net.node_ids/net.node_map for the
    # same input graph -- but without pyvis's own from_nx()/add_node()/
    # add_edge(), which check membership via `x in self.node_ids` and
    # `x in self.edges` (plain Python lists, not sets), an O(V) or O(E)
    # scan on every single node/edge insertion. At real large-graph scale
    # that's catastrophic: confirmed directly, from_nx() alone took 174s
    # at just 10,000 nodes/50,000 edges. A plain nx.Graph can't have
    # duplicate edges between the same pair, and every node here is added
    # exactly once by the caller, so none of pyvis's duplicate-checking is
    # actually needed.
    for node_id, data in graph.nodes(data=True):
        opts = dict(data)
        opts.setdefault("size", default_node_size)
        opts["size"] = int(opts["size"])
        label = opts.pop("label", None) or node_id
        shape = opts.pop("shape", "dot")
        color = opts.pop("color", "#97c2fc")
        if "group" in opts:
            n = Node(node_id, shape, label=label, font_color=net.font_color, **opts)
        else:
            n = Node(node_id, shape, label=label, color=color, font_color=net.font_color, **opts)
        net.nodes.append(n.options)
        net.node_ids.append(node_id)
        net.node_map[node_id] = n.options

    for source, target, data in graph.edges(data=True):
        opts = dict(data)
        if "weight" in opts and "width" not in opts:
            opts["width"] = opts.pop("weight")
        elif "weight" not in opts and "width" not in opts:
            opts["width"] = default_edge_weight
        e = Edge(source, target, net.directed, **opts)
        net.edges.append(e.options)


def render_network(graph, out_path, notebook=False, inject_filters=_inject_filter_controls,
                    physics=True):
    # show_buttons() would pull in its own control-panel styling the same way
    # -- omitted so the output file stays self-contained.
    net = Network(height="800px", width="100%", notebook=notebook, cdn_resources="in_line")
    net.set_options(_NETWORK_OPTIONS_JSON if physics else _STATIC_NETWORK_OPTIONS_JSON)
    _fast_populate_network(net, graph)
    net.write_html(out_path, notebook=notebook)
    _strip_bootstrap_cdn(out_path)
    inject_filters(out_path, graph)
    if physics:
        _inject_stabilize_then_stop(out_path)
    print(f"  wrote {out_path} ({graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges)")
    return net

graph = build_bipartite_graph(field_tables, seed_tags)
net = render_network(graph, NETWORK_OUT, notebook=True)
print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges -> {NETWORK_OUT}")
iframe = net.show(NETWORK_OUT, notebook=True)
# net.show() internally calls write_html() again -- a full raw rewrite from
# pyvis's un-postprocessed template -- which would silently wipe out
# _strip_bootstrap_cdn's, _inject_filter_controls', and
# _inject_stabilize_then_stop's edits. Re-run them on the file it just
# wrote. iframe only stores a src= path resolved lazily by the browser, so
# reprocessing the file before the cell ends is safe.
_strip_bootstrap_cdn(NETWORK_OUT)
_inject_filter_controls(NETWORK_OUT, graph)
_inject_stabilize_then_stop(NETWORK_OUT)
iframe


## Co-occurrence heatmaps

One heatmap per field: rows = seed tags, columns = attribute values, cell = % of that seed tag's works (color and displayed number both). A seed tag with no surviving edges for a field (e.g. filtered out by `MIN_COUNT` or `MIN_PROPORTION`) still appears as an all-zero row rather than disappearing.

In [ ]:
def cooccurrence_matrix(counts, field, seed_tags, normalize_by=None):
    if counts.empty:
        return pd.DataFrame(index=seed_tags)
    matrix = counts.pivot(index="tag", columns=field, values="count")
    matrix = matrix.fillna(0)
    columns = sorted(matrix.columns)
    matrix = matrix.reindex(index=seed_tags, columns=columns, fill_value=0)
    if normalize_by is not None:
        matrix = matrix.div(normalize_by.reindex(seed_tags), axis=0) * 100
    else:
        matrix = matrix.astype(int)
    return matrix


def render_heatmap(matrix, field, out_path, normalized=False, cmap="viridis",
                    center=None, xlabel=None, ylabel=None, cbar_label=None, fmt=None):
    if matrix.empty or matrix.shape[1] == 0:
        print(f"  skipping heatmap for {field}: no data after filtering")
        return
    width = max(8, 0.35 * matrix.shape[1] + 3)
    height = max(6, 0.22 * matrix.shape[0] + 3)
    fig, ax = plt.subplots(figsize=(width, height))
    annot = matrix.shape[0] * matrix.shape[1] <= 500
    if fmt is None:
        fmt = ".1f" if normalized else "d"
    if cbar_label is None:
        cbar_label = "% of tag's works" if normalized else "co-occurrence count"
    # mask=matrix.isna() is a no-op for the existing pipeline's matrices (never
    # contain NaN); the tag-pair matrix uses NaN for "no data" (as opposed to 0,
    # a meaningful independence value), rendered as blank cells.
    sns.heatmap(matrix, annot=annot, fmt=fmt, cmap=cmap, center=center,
                mask=matrix.isna(), cbar_kws={"label": cbar_label}, ax=ax)
    ax.set_xlabel(xlabel if xlabel is not None else field)
    ax.set_ylabel(ylabel if ylabel is not None else "seed tag")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"  wrote {out_path} ({matrix.shape[0]}x{matrix.shape[1]})")


os.makedirs(HEATMAP_OUT_DIR, exist_ok=True)
# work_counts was already computed in the cell above (shared with the
# MIN_PROPORTION filter). Heatmap cells always show/color by percentage of
# each seed tag's works.
for field, counts in field_tables.items():
    matrix = cooccurrence_matrix(counts, field, seed_tags, normalize_by=work_counts)
    out_path = os.path.join(HEATMAP_OUT_DIR, f"heatmap_{field}.png")
    render_heatmap(matrix, field, out_path, normalized=True)

## Tag-pair co-occurrence (lift/PMI)

A different question from the rest of this notebook: not "which attribute values
does a seed tag associate with", but "which pairs of tags -- pooled across
`fandom`/`relationship`/`character`/`additional_tags` -- statistically tend to
co-occur (or avoid each other) more than chance would predict". Raw co-occurrence
counts conflate "both tags are individually common" with "these two tags are
actually associated"; lift and PMI correct for that:

- `lift(A, B) = P(A, B) / (P(A) * P(B))`
- `pmi(A, B) = log2(lift(A, B))`

`pmi > 0` means the pair co-occurs *more* than chance ("most likely"), `pmi < 0`
means *less* than chance ("least likely"), `pmi == 0` means independence. This is
opt-in (`COMPUTE_TAG_PAIRS = False` by default in the Configuration cell above) --
a full pairwise computation across thousands of possible tags would otherwise slow
down every re-run.


In [ ]:
def build_document_tag_table(df, fields=TAG_PAIR_FIELDS):
    """Long (work_id, tag_id) table, one row per (distinct work, namespaced
    tag). A work matching multiple seed-tag searches gets a separate row
    per seed tag in df, all sharing the same work_id and identical
    fandom/relationship/character/additional_tags content (same AO3 work
    page) -- so df is deduplicated to one row per work_id first; this
    analysis doesn't care which seed tag(s) found a work, only its own
    content. tag_id is namespaced f"{field}::{value}" (same scheme
    build_bipartite_graph uses) so e.g. a fandom and a character sharing a
    literal name are never conflated."""
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    tables = []
    for field in fields:
        exploded = explode_field(deduped, field)
        exploded = exploded.rename(columns={field: "value"})
        exploded["tag_id"] = field + "::" + exploded["value"]
        tables.append(exploded[["work_id", "tag_id"]])
    return pd.concat(tables, ignore_index=True)


def top_k_tags_by_document_frequency(tag_table, k):
    """Analogous to top_n_values, but over the unified tag pool at the
    work/document level (tag_table has at most one row per (work_id,
    tag_id), so counting rows == counting documents). Returns None (no
    filtering) if k is None. Ties broken alphabetically, same convention
    as top_n_values."""
    if k is None:
        return None
    counts = tag_table["tag_id"].value_counts().reset_index()
    counts.columns = ["tag_id", "count"]
    counts = counts.sort_values(["count", "tag_id"], ascending=[False, True])
    return set(counts["tag_id"].head(k))


def build_tag_incidence_matrix(tag_table, keep_tags):
    """Documents x tags boolean incidence matrix (index=work_id,
    columns=sorted keep_tags, int8). NOTE: its row count is only documents
    containing >=1 kept tag -- it is NOT the total document count and must
    never be used as n_docs (a work with none of the top-K tags has no row
    here, but still belongs in P(A)'s denominator). Column sums (per-tag
    totals) ARE always correct regardless of this row restriction."""
    filtered = tag_table[tag_table["tag_id"].isin(keep_tags)]
    columns = sorted(keep_tags)
    work_ids = sorted(filtered["work_id"].unique())
    col_index = {tag_id: i for i, tag_id in enumerate(columns)}
    row_index = {work_id: i for i, work_id in enumerate(work_ids)}

    row_idx = filtered["work_id"].map(row_index).to_numpy()
    col_idx = filtered["tag_id"].map(col_index).to_numpy()
    matrix = sp.coo_matrix(
        (np.ones(len(filtered), dtype=np.int8), (row_idx, col_idx)),
        shape=(len(work_ids), len(columns)), dtype=np.int8,
    ).tocsr()
    # Collapse any duplicate (work_id, tag_id) rows to a plain 0/1 presence
    # flag -- matches pd.crosstab(...) > 0's semantics regardless of raw
    # counts (coo_matrix sums duplicate entries on conversion to csr).
    matrix.data[:] = 1
    return pd.DataFrame.sparse.from_spmatrix(matrix, index=work_ids, columns=columns)


def tag_pair_statistics(incidence, n_docs):
    """Fully vectorized pairwise lift/PMI -- no nested Python loop over
    pairs. joint = incidence.T @ incidence gives the whole tags x tags
    co-occurrence matrix in one matmul; its diagonal is each tag's own
    total document count (marginal). n_docs must be the TRUE total document
    count (df["work_id"].nunique() on the original, undeduplicated df --
    nunique() already ignores duplicate rows), never incidence.shape[0]
    (see build_tag_incidence_matrix) -- using the latter would silently
    undercount the corpus and skew every lift/PMI value.

    lift(A,B) = joint_count * n_docs / (count_a * count_b); pmi =
    log2(lift). Pairs with joint_count == 0 are dropped before the log2
    call (never assigned -inf) -- they carry no real co-occurrence signal.

    Returns a long DataFrame [tag_a, tag_b, joint_count, count_a, count_b,
    lift, pmi], one row per unordered pair with tag_a < tag_b, joint_count
    > 0.
    """
    tags = list(incidence.columns)
    values = incidence.sparse.to_coo().tocsr().astype(np.int64)
    joint = values.T @ values
    marginal = np.asarray(joint.diagonal()).ravel()

    # scipy.sparse.triu only stores the upper triangle's nonzero entries --
    # equivalent to np.triu_indices(k=1) on the dense matrix, but without
    # ever materializing the full tags x tags array to get there (a dense
    # tags x tags int64 array is 48.9 GiB at 81,037 tags -- exactly the
    # MemoryError this sparse rewrite fixes).
    upper = sp.triu(joint, k=1).tocoo()
    i_idx, j_idx, joint_counts = upper.row, upper.col, upper.data
    nonzero = joint_counts > 0
    i_idx, j_idx, joint_counts = i_idx[nonzero], j_idx[nonzero], joint_counts[nonzero]

    count_a = marginal[i_idx]
    count_b = marginal[j_idx]
    lift = (joint_counts * n_docs) / (count_a * count_b)
    pmi = np.log2(lift)

    # Explicit canonicalization: tag_a is always the alphabetically-smaller
    # of the pair, regardless of the incidence matrix's own column order --
    # np.triu_indices above only guarantees i < j by column *position*, not
    # by alphabetical value. This used to be true only as an incidental side
    # effect of build_tag_incidence_matrix always pre-sorting its columns;
    # enforcing it here means the guarantee holds for any caller, not just
    # today's one. lift/pmi are symmetric in count_a/count_b already, so
    # only the names and their paired counts need to swap together.
    tags_arr = np.array(tags)
    a_names = tags_arr[i_idx]
    b_names = tags_arr[j_idx]
    swap = a_names > b_names
    tag_a = np.where(swap, b_names, a_names)
    tag_b = np.where(swap, a_names, b_names)
    count_a_final = np.where(swap, count_b, count_a)
    count_b_final = np.where(swap, count_a, count_b)

    return pd.DataFrame({
        "tag_a": tag_a,
        "tag_b": tag_b,
        "joint_count": joint_counts,
        "count_a": count_a_final,
        "count_b": count_b_final,
        "lift": lift,
        "pmi": pmi,
    })


def apply_min_pair_count(pair_stats, min_pair_count):
    """Analogous to apply_min_count. Must run before apply_pmi_thresholds --
    this is what keeps a pair like (count_a=1, count_b=1, joint=1), whose
    lift is an enormous but statistically meaningless n_docs, out of the
    ranked/thresholded output."""
    return pair_stats[pair_stats["joint_count"] >= min_pair_count]


def apply_pmi_thresholds(pair_stats, min_pmi, max_pmi):
    """Keeps a pair iff pmi >= min_pmi (co-occurs more than chance, "most
    likely") OR pmi <= max_pmi (co-occurs less than chance, "least
    likely"), dropping the "boring middle" near independence."""
    return pair_stats[(pair_stats["pmi"] >= min_pmi) | (pair_stats["pmi"] <= max_pmi)]


def tag_pair_matrix(pair_stats, tags, value_col="pmi"):
    """Symmetric tag x tag DataFrame (index=columns=sorted(tags)). Cells
    for pairs absent from pair_stats (filtered out upstream, or originally
    joint_count==0) are NaN, not 0 -- 0 is itself a meaningful PMI value
    (independence) and must not be confused with "no data". The diagonal
    is always NaN. tags is the full top-K keep_tags set (not just tags
    that survived pair filtering), so a tag with zero surviving pairs
    still appears as an all-NaN row/column, mirroring the existing "Found
    Family stays as an all-zero row" precedent for the seed-tag heatmaps."""
    ordered = sorted(tags)
    matrix = pd.DataFrame(np.nan, index=ordered, columns=ordered)
    for _, row in pair_stats.iterrows():
        if row["tag_a"] in matrix.index and row["tag_b"] in matrix.index:
            matrix.loc[row["tag_a"], row["tag_b"]] = row[value_col]
            matrix.loc[row["tag_b"], row["tag_a"]] = row[value_col]
    return matrix


def build_tag_pair_graph(pair_stats):
    """One-mode (non-bipartite) graph: node id = the tag_id itself (e.g.
    "fandom::Alpha"), label = the part after "::", group/color = the part
    before "::" (field recovered from the id -- no separate lookup table
    needed, since namespacing already encodes it)."""
    graph = nx.Graph()
    for _, row in pair_stats.iterrows():
        for tag_id in (row["tag_a"], row["tag_b"]):
            if tag_id not in graph:
                field, _, value = tag_id.partition("::")
                graph.add_node(tag_id, label=value, group=field,
                                color=FIELD_COLORS[field], title=tag_id)
        relation = "more likely than chance" if row["pmi"] > 0 else "less likely than chance"
        edge_color = MOST_LIKELY_EDGE_COLOR if row["pmi"] > 0 else LEAST_LIKELY_EDGE_COLOR
        graph.add_edge(
            row["tag_a"], row["tag_b"], weight=int(row["joint_count"]),
            pmi=float(row["pmi"]), lift=float(row["lift"]),
            joint_count=int(row["joint_count"]), color=edge_color,
            title=(f"{row['tag_a']} × {row['tag_b']}: {relation} "
                   f"(lift={row['lift']:.2f}, pmi={row['pmi']:.2f}, "
                   f"joint count={int(row['joint_count'])})"),
        )
    return graph


def build_tag_pair_data(df, top_tags, min_pair_count, min_pmi, max_pmi):
    """Orchestrator, analogous to build_field_data. Returns (pair_stats,
    keep_tags) -- keep_tags is the full top-K tag universe (needed by
    tag_pair_matrix separately from pair_stats, which only has surviving
    pairs)."""
    tag_table = build_document_tag_table(df)
    keep_tags = top_k_tags_by_document_frequency(tag_table, top_tags)
    if keep_tags is None:
        keep_tags = set(tag_table["tag_id"].unique())
    incidence = build_tag_incidence_matrix(tag_table, keep_tags)
    n_docs = df["work_id"].nunique()
    pair_stats = tag_pair_statistics(incidence, n_docs)
    pair_stats = apply_min_pair_count(pair_stats, min_pair_count)
    pair_stats = apply_pmi_thresholds(pair_stats, min_pmi, max_pmi)
    return pair_stats, keep_tags


In [ ]:
if COMPUTE_TAG_PAIRS:
    pair_stats, keep_tags = build_tag_pair_data(df, TOP_TAGS, MIN_PAIR_COUNT, MIN_PMI, MAX_PMI)

    print("Building tag-pair network graph")
    tag_pair_graph = build_tag_pair_graph(pair_stats)
    tag_pair_net = render_network(tag_pair_graph, TAG_PAIR_NETWORK_OUT, notebook=True,
                                   inject_filters=_inject_tag_pair_filter_controls)
    print(f"{tag_pair_graph.number_of_nodes()} nodes, "
          f"{tag_pair_graph.number_of_edges()} edges -> {TAG_PAIR_NETWORK_OUT}")
    tag_pair_iframe = tag_pair_net.show(TAG_PAIR_NETWORK_OUT, notebook=True)
    # net.show() internally calls write_html() again, which would wipe out the
    # postprocessing below -- re-run it on the file it just wrote (same dance as
    # the seed-tag network cell above).
    _strip_bootstrap_cdn(TAG_PAIR_NETWORK_OUT)
    _inject_tag_pair_filter_controls(TAG_PAIR_NETWORK_OUT, tag_pair_graph)
    _inject_stabilize_then_stop(TAG_PAIR_NETWORK_OUT)

    print("Building tag-pair heatmap")
    os.makedirs(os.path.dirname(TAG_PAIR_HEATMAP_OUT) or ".", exist_ok=True)
    tag_pair_matrix_df = tag_pair_matrix(pair_stats, keep_tags, value_col="pmi")
    render_heatmap(tag_pair_matrix_df, "tag pair", TAG_PAIR_HEATMAP_OUT, cmap="coolwarm",
                   center=0, xlabel="tag", ylabel="tag", cbar_label="PMI (log2 lift)", fmt=".2f")

    tag_pair_iframe
else:
    print("Skipped (COMPUTE_TAG_PAIRS is False) -- set it to True in the Configuration "
          "cell to compute/render tag-pair co-occurrence stats.")


## Done

`{NETWORK_OUT}` and `{HEATMAP_OUT_DIR}/heatmap_<field>.png` (one per field in
`FIELDS_TO_VISUALIZE`) are now in the notebook's working directory. Re-run from the
Configuration cell with different `TOP_FANDOMS`/`TOP_ADDITIONAL_TAGS`/`MIN_COUNT`
(or `MIN_PROPORTION`)/`TOP_SEED_TAGS` values to adjust what's included.

If `COMPUTE_TAG_PAIRS` was set to `True`, `{TAG_PAIR_NETWORK_OUT}` and `{TAG_PAIR_HEATMAP_OUT}` are also written -- the tag-to-tag lift/PMI network and heatmap. Re-run from the Configuration cell with different `TOP_TAGS`/`MIN_PAIR_COUNT`/`MIN_PMI`/`MAX_PMI` values to adjust what's included.